In [1]:
import pandas as pd
import numpy as np

In [2]:
rain = pd.read_parquet('rr_monthly_all.parquet')

In [3]:
temp = pd.read_parquet('tg_monthly_all.parquet')

In [4]:
rain

,time,latitude,longitude,rr
0,1950-01-01,25.049861,-24.95014,0.0
1,1950-01-01,25.049861,-24.85014,0.0
2,1950-01-01,25.049861,-24.75014,0.0
3,1950-01-01,25.049861,-24.65014,0.0
4,1950-01-01,25.049861,-24.55014,0.0
...,...,...,...,...
295042495,2024-12-01,71.449859,45.04986,0.0
295042496,2024-12-01,71.449859,45.14986,0.0
295042497,2024-12-01,71.449859,45.24986,0.0
295042498,2024-12-01,71.449859,45.34986,0.0


In [5]:
temp

,time,latitude,longitude,tg
0,1950-01-01,25.049861,-14.75014,16.254519
1,1950-01-01,25.049861,-14.65014,16.410322
2,1950-01-01,25.049861,-14.55014,16.418064
3,1950-01-01,25.049861,-14.45014,16.397741
4,1950-01-01,25.049861,-14.35014,16.635483
...,...,...,...,...
116907885,2024-12-01,70.949860,28.04986,-3.646128
116907886,2024-12-01,70.949860,28.14986,-3.280645
116907887,2024-12-01,70.949860,28.24986,-4.229355
116907888,2024-12-01,70.949860,28.34986,-3.966129


In [6]:
join_keys = ['time', 'latitude', 'longitude']

# Perform the inner join
# The 'on' parameter takes the list of columns
# The 'how' parameter specifies the type of join
df = pd.merge(rain, temp, on=join_keys, how='inner')

In [7]:
df.to_parquet("europe.parquet")

In [8]:
df

,time,latitude,longitude,rr,tg


In [9]:
from __future__ import annotations
from geopy.geocoders import Nominatim
from dataclasses import dataclass
import xarray as xr
import pandas as pd
import numpy as np

@dataclass
class Location:
    name: str
    latitude: float
    longitude: float

_geolocator = None

def geocode_city(city: str, user_agent: str = "pogoda-app") -> Location:
    """Geocode a city name into coordinates using Nominatim.

    Parameters
    ----------
    city: str
        City name (can include country, e.g., "Szczecin, Poland").
    user_agent: str
        Required by Nominatim usage policy.

    Returns
    -------
    Location

    Raises
    ------
    ValueError: if no result is found.
    """
    global _geolocator
    if _geolocator is None:
        _geolocator = Nominatim(user_agent=user_agent, timeout=10)
    result = _geolocator.geocode(city)
    if not result:
        raise ValueError(f"City not found: {city}")
    return Location(name=result.address, latitude=result.latitude, longitude=result.longitude)


In [11]:
city = geocode_city("trieste")

In [14]:
from sklearn.neighbors import BallTree

In [20]:
lat_rad = np.deg2rad(temp['latitude'].values)
lon_rad = np.deg2rad(temp['longitude'].values)

In [21]:
points = np.vstack([lat_rad, lon_rad]).T

In [22]:
points

array([[ 0.43720254, -0.2574385 ],
       [ 0.43720254, -0.25569317],
       [ 0.43720254, -0.25394784],
       ...,
       [ 1.23830867,  0.49305308],
       [ 1.23830867,  0.4947984 ],
       [ 1.23830867,  0.49654373]])

In [23]:
tree = BallTree(points, metric='haversine')

In [24]:
# Convert the target location to radians
target_point_rad = np.deg2rad([[city.latitude, city.longitude]])

In [25]:
# Query the tree for the nearest neighbor
# k=1 means we are looking for the single closest point
distance, index = tree.query(target_point_rad, k=3)

In [26]:
distance

array([[0.00088555, 0.00088555, 0.00088555]])

In [27]:
index

array([[293031,  37905, 165468]])

In [29]:
place = temp.iloc[index[0][0]]

In [30]:
place

time         1950-03-01 00:00:00
latitude               45.649861
longitude               13.84986
tg                      8.482903
Name: 293031, dtype: object

In [33]:
temp[[temp.latitude==place.latitude]]

KeyError: "None of [Index([(False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, ...)], dtype='object')] are in the [columns]"

In [34]:
temp.groupby(['latitude', 'longitude']).size().shape[0]

136070